<a href="https://colab.research.google.com/github/devesssi/llm-diffusion-models-finetuning/blob/main/non_instructional(finetuning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U peft bitsandbytes transformers accelerate

In [ ]:
!pip install -U trl

In [ ]:
!pip install PyMuPDF

In [ ]:
import fitz

In [ ]:
def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

In [ ]:
pdf_texts = extract_text_from_pdf("/content/THE INDUSTRIAL FRONTIER.pdf")

In [ ]:
pdf_texts

['THE INDUSTRIAL FRONTIER: STRUCTURAL LEVERAGE \n• \nAndrew Carnegie | The Rule of Vertical Integration: Eliminate external \ndependencies and transaction costs by owning every node of the supply chain.  \no Example Use Case: Carnegie bypassed middlemen by purchasing the \nScotia Ore Mines, fleets of ore boats, and the Union Railway to feed his \nsteel mills. This permanently lowered his cost basis and secured \nindependence from suppliers, granting him absolute structural \nsovereignty. \n• \nJohn D. Rockefeller | The Rule of Horizontal Consolidation & Precision \nAccounting: Expand by acquiring competitors at the same production stage, and \nutilize extreme cost accounting to reveal invisible systemic bottlenecks.  \no Example Use Case: During the 1873 economic depression, Rockefeller \nexploited the crisis to buy out bankrupt rival refineries at distressed \nprices. He tracked costs "to the third decimal" and leveraged his massive \nvolume to negotiate exclusive railroad rebates, co

In [ ]:
import re
def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        # Split on double line breaks or long newlines
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:  # ignore too short lines
                paragraphs.append(clean)
    return paragraphs

In [ ]:
paragraphs = split_paragraphs(pdf_texts)

In [ ]:
data = [{"text": p} for p in paragraphs]

In [ ]:
print(data)

[{'text': 'THE INDUSTRIAL FRONTIER: STRUCTURAL LEVERAGE \n• \nAndrew Carnegie | The Rule of Vertical Integration: Eliminate external \ndependencies and transaction costs by owning every node of the supply chain.  \no Example Use Case: Carnegie bypassed middlemen by purchasing the \nScotia Ore Mines, fleets of ore boats, and the Union Railway to feed his \nsteel mills. This permanently lowered his cost basis and secured \nindependence from suppliers, granting him absolute structural \nsovereignty. \n• \nJohn D. Rockefeller | The Rule of Horizontal Consolidation & Precision \nAccounting: Expand by acquiring competitors at the same production stage, and \nutilize extreme cost accounting to reveal invisible systemic bottlenecks.  \no Example Use Case: During the 1873 economic depression, Rockefeller \nexploited the crisis to buy out bankrupt rival refineries at distressed \nprices. He tracked costs "to the third decimal" and leveraged his massive \nvolume to negotiate exclusive railroad re

In [ ]:
from datasets import Dataset, load_dataset

In [ ]:
dataset = Dataset.from_list(data)

In [ ]:
dataset

Dataset({
    features: ['text'],
    num_rows: 3
})

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(examples):
    tokens = tokenizer(examples["text"],truncation=True,padding="max_length",max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [ ]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [ ]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!pip install -U peft bitsandbytes transformers accelerate

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset


In [ ]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model)

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [ ]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [ ]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model,
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)


In [ ]:
non_inst_model_lora = get_peft_model(model, lora_config)

In [ ]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=tokenized
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss


TrainOutput(global_step=5, training_loss=2.5564674377441405, metrics={'train_runtime': 12.9276, 'train_samples_per_second': 1.16, 'train_steps_per_second': 0.387, 'total_flos': 47722235166720.0, 'train_loss': 2.5564674377441405, 'epoch': 5.0})

In [ ]:
model_path = "./llama-bill-lora"
non_inst_model_lora.save_pretrained(model_path)

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, model_path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
prompt = "Sam Altman | The Scaling Laws of Compounding: Treat resource growth as an"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.cuda()

output = model.generate(input_ids, max_new_tokens=50, num_return_sequences=1)

print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Sam Altman | The Scaling Laws of Compounding: Treat resource growth as an investment
The Scaling Laws of Compounding: Treat resource growth as an investment
Sam Altman | The Scaling Laws of Compounding: Treat resource growth as an investment | The Scaling La


In [ ]:
prompt = 'state some laws use by john rockefellar'
input_ids = tokenizer(prompt, return_tensors= 'pt').input_ids.cuda()

output = model.generate(input_ids, max_new_tokens=70, num_return_sequences=1)

print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=70) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


state some laws use by john rockefellar.
The 1935 law was passed by the 73rd Congress, and was signed into law by President Franklin D. Roosevelt on March 2, 1935.
The 1935 law was passed by the 73rd Congress, and was signed into law by President Franklin
